# Building Age Classification from Satellite Imagery

# Notebook 4 — Modelling & Evaluation

---

**Approach: Metric Learning**

In this notebook we will use a **metric learning** approach to the classification problem. This involves learning a feature representation where
buildings of the same age class cluster together, regardless of which city they come from. 

In practice this means:
1. **Train on Madrid** (the labelled city) using cross-validation to get reliable performance
   estimates with error bars.
2. **Zero-shot transfer** — apply the Madrid model directly to Amsterdam, with no Amsterdam
   labels, to measure the baseline domain gap.
3. **Few-shot prototype transfer** — given a small number of labelled Amsterdam examples,
   compute per-class prototype vectors (class means in feature space) and classify by
   nearest prototype. This is the inference step of prototype networks, a standard
   metric learning technique.

The few-shot experiment shows how much label budget is needed to close the Madrid–Amsterdam gap.

For more information on the metric learning approach, please see the supportind document "METRIC_LEARNING_APPROACH.md"

**NOTE**:  Metric learning is _one possible approach_.  Feel free to try another approach!

## 4.1 Setup & Imports

In [5]:
import pickle
import optuna
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from xgboost import XGBClassifier

CLASS_COLORS = {1: '#4393c3', 2: '#2ca25f', 3: '#fd8d3c', 4: '#de2d26'}
CLASS_LABELS_MADRID    = {1: 'Pre-1960',  2: '1960-1984', 3: '1984-2004', 4: '2004-2024'}
CLASS_LABELS_AMSTERDAM = {1: 'Pre-1945',  2: '1945-1984', 3: '1984-2004', 4: '2004-2024'}

plt.rcParams['figure.dpi'] = 100
print("Setup complete.")

# Turn off Optuna logging noise 
optuna.logging.set_verbosity(optuna.logging.WARNING)

Setup complete.


## 4.2 Load Preprocessed Data

In [2]:
with open('data/preprocessed/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_madrid    = data['X_madrid']
y_madrid    = data['y_madrid']
X_amsterdam = data['X_amsterdam']
y_amsterdam = data['y_amsterdam']
feature_names = data['feature_names']
pixel_ids_madrid    = data['pixel_ids_madrid']     # (N, 2)  px_key, py_key
pixel_ids_amsterdam = data['pixel_ids_amsterdam']  # (N, 2)  px_key, py_key

smote = SMOTE(random_state=42)
X_madrid, y_madrid = smote.fit_resample(X_madrid, y_madrid)

print(f'Madrid:    {X_madrid.shape}  ({len(np.unique(y_madrid))} classes)')
print(f'Amsterdam: {X_amsterdam.shape}  ({len(np.unique(y_amsterdam))} classes)')
print(f'Features:  {len(feature_names)}')
print()
for name, y, labels in [('Madrid', y_madrid, CLASS_LABELS_MADRID),
                         ('Amsterdam', y_amsterdam, CLASS_LABELS_AMSTERDAM)]:
    print(f'{name} class distribution:')
    for cls in sorted(np.unique(y)):
        n = (y == cls).sum()
        print(f'  Class {cls} ({labels[cls]}): {n:,}  ({n/len(y)*100:.1f}%)')
    print()
    

y_madrid = y_madrid - 1
y_amsterdam = y_amsterdam - 1

Madrid:    (108608, 60)  (4 classes)
Amsterdam: (25992, 60)  (4 classes)
Features:  60

Madrid class distribution:
  Class 1 (Pre-1960): 27,152  (25.0%)
  Class 2 (1960-1984): 27,152  (25.0%)
  Class 3 (1984-2004): 27,152  (25.0%)
  Class 4 (2004-2024): 27,152  (25.0%)

Amsterdam class distribution:
  Class 1 (Pre-1945): 7,568  (29.1%)
  Class 2 (1945-1984): 8,802  (33.9%)
  Class 3 (1984-2004): 6,745  (26.0%)
  Class 4 (2004-2024): 2,877  (11.1%)



In [3]:
NUM_CLASSES = len(np.unique(y_madrid))

### XGBOOST BASELINE MODEL

In [4]:
# create a baseline xgboost model 
results = {}

xgb_params = {
    "objective": "multi:softprob",
    "num_class": NUM_CLASSES,
    "max_depth": 5,
    "learning_rate": 0.05,
    "n_estimators": 150,
    "random_state": 42,
    "eval_metric": "mlogloss"
}

print("CROSS-VALIDATION ON MADRID ONLY")
print("=== 5-Fold Cross-Validation on Madrid ===")
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

madrid_cv_f1s = []

for fold, (train_ids, val_ids) in enumerate(cv.split(X_madrid, y_madrid)):
    X_train, X_val = X_madrid[train_ids], X_madrid[val_ids]
    y_train, y_val = y_madrid[train_ids], y_madrid[val_ids]

    # Train the fold model 
    fold_model = XGBClassifier(**xgb_params)
    fold_model.fit(X_train, y_train)

    # Predict and calculate fold Macro F1 
    y_pred = fold_model.predict(X_val)
    fold_f1 = f1_score(y_val, y_pred, average="macro")
    madrid_cv_f1s.append(fold_f1)
    print(f"Madrid Fold {fold + 1} Macro F1: {fold_f1:.4f}")

print(f"Mean Madrid CV Macro F1: {np.mean(madrid_cv_f1s):.4f} +/- {np.std(madrid_cv_f1s):.4f}\n")

print("ZERO-SHOT TRANSFER TO AMSTERDAM")
print("=== Evaluating Zero-Shot Transfer to Amsterdam ===")

# Train the model on all the data 
model = XGBClassifier(**xgb_params)
model.fit(X_madrid, y_madrid)

# Predict on the entirely untrained Amsterdam dataset 
y_pred_zero = model.predict(X_amsterdam)
zero_shot_f1 = f1_score(y_amsterdam, y_pred_zero, average="macro")
results[0] = zero_shot_f1
print(f"Zero-Shot Amsterdam Macro F1: {zero_shot_f1:.4f}\n")

# FEW-SHOT TRANSFER LOOP 
few_shot_sizes = [5, 10, 25, 50, 100, 200]

for n in few_shot_sizes:
    if n >= len(X_amsterdam):
        print(f"Skipping n={n}: Not enough data points in target dataset")
        continue

    # Split the data into train and test sets 
    X_shot, X_test_val, y_shot, y_test_val = train_test_split(
        X_amsterdam, y_amsterdam,
        train_size=n,
        stratify=y_amsterdam,
        random_state=42
    )

    # Use a regularized, low-capacity model setup to prevent few-shot overfitting 
    few_shot_model = XGBClassifier(
        objective="multi:softprob",
        num_class=NUM_CLASSES,
        max_depth=3, # Lower tree depth protects small n distributions
        learning_rate=0.05,
        n_estimators=80,   # Fewer iterations limit noise memorization
        random_state=42,
        eval_metric="mlogloss"
    )

    # Train directly on the minorr target slice 
    few_shot_model.fit(X_shot, y_shot)

    # Evaluate on the massive remainder of the target domain data 
    y_pred_shot = few_shot_model.predict(X_test_val)
    few_shot_f1 = f1_score(y_test_val, y_pred_shot, average="macro")

    results[n] = few_shot_f1
    print(f"Few-Shot (n={n:3d}) Amsterdam Macro F1: {few_shot_f1:.4f}")

# FINAL RESULTS CONSOLIDATION 
print(f"\n=== FINAL REUSE & ADAPTATION BENCHMARK SUMMARY ===")
print(f"Madrid Source CV Macro F1: {np.mean(madrid_cv_f1s):.4f}")
print("-"*45)
print(f"{'Target Scenario':<25}{'Amsterdam Macro F1 Score':<20}")
print("-"*45)
for shot, score in results.items():
    label = "Zero-Shot (No tuning)" if shot == 0 else f"{shot}-Shot Target Split"
    print(f"{label:<25}{score:.4f}")

CROSS-VALIDATION ON MADRID ONLY
=== 5-Fold Cross-Validation on Madrid ===
Madrid Fold 1 Macro F1: 0.6232
Madrid Fold 2 Macro F1: 0.6195
Madrid Fold 3 Macro F1: 0.6163
Madrid Fold 4 Macro F1: 0.6174
Madrid Fold 5 Macro F1: 0.6162
Mean Madrid CV Macro F1: 0.6185 +/- 0.0026

ZERO-SHOT TRANSFER TO AMSTERDAM
=== Evaluating Zero-Shot Transfer to Amsterdam ===
Zero-Shot Amsterdam Macro F1: 0.4840

Few-Shot (n=  5) Amsterdam Macro F1: 0.1265
Few-Shot (n= 10) Amsterdam Macro F1: 0.3857
Few-Shot (n= 25) Amsterdam Macro F1: 0.5579
Few-Shot (n= 50) Amsterdam Macro F1: 0.5959
Few-Shot (n=100) Amsterdam Macro F1: 0.6038
Few-Shot (n=200) Amsterdam Macro F1: 0.6438

=== FINAL REUSE & ADAPTATION BENCHMARK SUMMARY ===
Madrid Source CV Macro F1: 0.6185
---------------------------------------------
Target Scenario          Amsterdam Macro F1 Score
---------------------------------------------
Zero-Shot (No tuning)    0.4840
5-Shot Target Split      0.1265
10-Shot Target Split     0.3857
25-Shot Target Spl

### XGBoost Hyperparameter Tuning

In [6]:
benchmark_results = {}

# HYPERPARAMETER TUNING & CROSS-VALIDATION ON MADRID 
print("=== Running Optuna Tuning via 5-Fold CV on Madrid ===")

def objective(trial):
    # Define hyperparameter search spaces for XGBoost 
    params = {
        "objective": "multi:softprob",
        "num_class": NUM_CLASSES,
        "eval_metric": "mlogloss",
        "random_state": 42,
        "n_estimators": trial.suggest_int('n_estimators', 50, 250),
        "max_depth": trial.suggest_int('max_depth', 3, 9),
        "learning_rate": trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        "subsample": trial.suggest_float('subsample', 0.6, 1.0),
        "colsample_bytree": trial.suggest_float('colsample_bytree', 0.6, 1.0),
        "min_child_weight": trial.suggest_int('min_child_weight', 1, 10),
        "gamma": trial.suggest_float('gamma', 0.0, 1.0)
    }

    cv = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=42
    )
    cv_f1s = []

    for train_ids, val_ids in cv.split(X_madrid, y_madrid):
        X_train, X_val = X_madrid[train_ids], X_madrid[val_ids]
        y_train, y_val = y_madrid[train_ids], y_madrid[val_ids]

        fold_model = XGBClassifier(**params)
        fold_model.fit(X_train, y_train)

        y_pred = fold_model.predict(X_val)
        cv_f1s.append(f1_score(y_val, y_pred, average="macro"))

    # Optuna maximizes this returned objective score 
    return np.mean(cv_f1s)

# Run optimization study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

best_params = study.best_params

# Inject static configuration required for multi-class structure 
best_params.update({
    "objective": "multi:softprob",
    "num_class": NUM_CLASSES,
    "eval_metric": "mlogloss",
    "random_state": 42
})

print(f"✔ Optuna Tuning Complete.")
print(f"Best Madrid CV Macro F1: {study.best_value:.4f}")
print(f"Optuna Parameters: {study.best_params}")

# ZERO-SHOT TRANSFER TO AMSTERDAM 
print("\n=== Evaluating Zero-Shot Transfer to Amsterdam ===")

# Train using the best params on 100% madrid data 
model = XGBClassifier(**best_params)
model.fit(X_madrid, y_madrid)

y_pred_zero = model.predict(X_amsterdam)
zero_shot_f1 = f1_score(y_amsterdam, y_pred_zero, average="macro")

benchmark_results[0] = zero_shot_f1
print(f"Zero-Shot Amsterdam Macro F1: {zero_shot_f1:.4f}\n")

# FEW-SHOT TRANSFER LOOP 
print("=== Evaluating Few-Shot Transfer eon Amsterdam ===")
few_shot_sizes = [5, 10, 25, 50, 100, 200]

for n in few_shot_sizes:
    if n >= len(X_amsterdam):
        continue
    X_shot, X_test_val, y_shot, y_test_val = train_test_split(
        X_amsterdam, y_amsterdam,
        train_size=n,
        stratify=y_amsterdam,
        random_state=42
    )

    # Adapt parameters dynamically for few-shot learning to prevent severe overfitting 
    # Copy Optuna params but enforce strict constraints for small n-datasets 
    few_shot_params = best_params.copy()
    few_shot_params["max_depth"] = min(3, best_params.get('max_depth', 3))
    few_shot_params["n_estimators"] = min(80, best_params.get("n_estimators", 80))

    few_shot_model = XGBClassifier(**few_shot_params)
    few_shot_model.fit(X_shot, y_shot)

    y_pred_shot = few_shot_model.predict(X_test_val)
    few_shot_f1 = f1_score(y_test_val, y_pred_shot, average="macro")

    benchmark_results[n] = few_shot_f1
    print(f"Few-Shot (n={n:3d}) Amsterdam Macro F1: {few_shot_f1:.4f}")   

# FINAL RESULTS CONSOLIDATION 
print("\n=== FINAL REUSE & ADAPTATION BENCHMARK SUMMARY ===")
print(f"Tuned Madrid Source CV Macro F1: {study.best_value:.4f}")
print("-"*45)
print(f"{'Target Scenario':<25}{'Amsterdam Macro F1 Score':<20}")
print("-"*45)
for shot, score in benchmark_results.items():
    label = "Zero-Shot (No tuning)" if shot == 0 else f"{shot}-Shot Target Split"
    print(f"{label:<25}{score:.4f}")

=== Running Optuna Tuning via 5-Fold CV on Madrid ===
✔ Optuna Tuning Complete.
Best Madrid CV Macro F1: 0.7505
Optuna Parameters: {'n_estimators': 211, 'max_depth': 9, 'learning_rate': 0.18452148500363144, 'subsample': 0.8662241995545219, 'colsample_bytree': 0.637025403352621, 'min_child_weight': 3, 'gamma': 0.014813293522783934}

=== Evaluating Zero-Shot Transfer to Amsterdam ===
Zero-Shot Amsterdam Macro F1: 0.4626

=== Evaluating Few-Shot Transfer eon Amsterdam ===
Few-Shot (n=  5) Amsterdam Macro F1: 0.1265
Few-Shot (n= 10) Amsterdam Macro F1: 0.1030
Few-Shot (n= 25) Amsterdam Macro F1: 0.4055
Few-Shot (n= 50) Amsterdam Macro F1: 0.5551
Few-Shot (n=100) Amsterdam Macro F1: 0.6026
Few-Shot (n=200) Amsterdam Macro F1: 0.6623

=== FINAL REUSE & ADAPTATION BENCHMARK SUMMARY ===
Tuned Madrid Source CV Macro F1: 0.7505
---------------------------------------------
Target Scenario          Amsterdam Macro F1 Score
---------------------------------------------
Zero-Shot (No tuning)    0.4